# 导入

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

#  1. Data Loading

In [2]:
RANDOM_STATE = 42
PCA_VAR_RATIO = 0.95

# t-SNE 为了 notebook 可控，先限制抽样
TSNE_SAMPLE_SIZE = 2500

# 是否真的运行 t-SNE
RUN_TSNE = True

# 每个算法/特征组是否都展示图
SHOW_ALL_BEST_RESULTS = True

In [3]:
file_path = "AirQuality_Fully_Preprocessed.csv"
df = pd.read_csv(file_path)

display(Markdown("## 原始数据基本信息"))
print("Shape:", df.shape)
display(df.head())

## 原始数据基本信息

Shape: (9357, 41)


,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),...,H_14,H_15,H_16,H_17,H_18,H_19,H_20,H_21,H_22,H_23
0,0.327869,1.177627,-1.613973,0.229342,0.387741,-0.371614,0.873072,0.072491,0.695541,0.582197,...,0,0,0,0,1,0,0,0,0,0
1,-0.091225,0.865965,-2.125772,-0.103840,0.048002,-0.679977,1.334557,-0.379513,0.307998,-0.149704,...,0,0,0,0,0,1,0,0,0,0
2,0.048473,1.370125,-2.449012,-0.157149,-0.011733,-0.542927,1.201587,0.094015,0.296343,0.102505,...,0,0,0,0,0,0,1,0,0,0
3,0.048473,1.250960,-2.556759,-0.130495,0.021868,-0.342246,1.013864,0.266206,0.380845,0.421476,...,0,0,0,0,0,0,0,1,0,0
4,-0.370621,0.774299,-2.947342,-0.490331,-0.396273,-0.542927,1.455795,0.137063,0.106942,0.191520,...,0,0,0,0,0,0,0,0,1,0


# 2. 定义特征组

In [4]:
pollutant_cols = [
    "CO(GT)", "NMHC(GT)", "C6H6(GT)", "NOx(GT)", "NO2(GT)"
]

sensor_cols = [
    "PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)", "PT08.S5(O3)"
]

meteo_cols = ["T", "RH", "AH"]

calendar_cols = ["Year", "Month", "DayOfWeek"]
hour_cols = [f"H_{i}" for i in range(24)]
temporal_cols = calendar_cols + hour_cols

feature_sets = {
    "all_features": list(df.columns),

    "air_quality_core": pollutant_cols + sensor_cols + meteo_cols,

    "air_quality_core_temporal": pollutant_cols + sensor_cols + meteo_cols + temporal_cols,

    # single
    """单独有CO, PTO8这样的 """
    """单独有 CO + PTO8  """

    "co_related": ["CO(GT)", "PT08.S1(CO)"] + meteo_cols + temporal_cols,
    "nmhc_related": ["NMHC(GT)", "PT08.S2(NMHC)"] + meteo_cols + temporal_cols,
    "nox_related": ["NOx(GT)", "PT08.S3(NOx)"] + meteo_cols + temporal_cols,
    "no2_related": ["NO2(GT)", "PT08.S4(NO2)"] + meteo_cols + temporal_cols,
    "benzene_related": ["C6H6(GT)", "PT08.S2(NMHC)"] + meteo_cols + temporal_cols,
}

# 去重 + 检查
for k, cols in feature_sets.items():
    unique_cols = []
    for c in cols:
        if c not in unique_cols:
            unique_cols.append(c)
    missing_cols = [c for c in unique_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{k} 中不存在这些列: {missing_cols}")
    feature_sets[k] = unique_cols

display(Markdown("## Feature Sets"))
feature_set_info = pd.DataFrame({
    "feature_set": list(feature_sets.keys()),
    "n_features": [len(v) for v in feature_sets.values()],
    "features": [", ".join(v[:8]) + (" ..." if len(v) > 8 else "") for v in feature_sets.values()]
})
display(feature_set_info)

## Feature Sets

,feature_set,n_features,features
0,all_features,41,"CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08...."
1,air_quality_core,13,"CO(GT), NMHC(GT), C6H6(GT), NOx(GT), NO2(GT), ..."
2,air_quality_core_temporal,40,"CO(GT), NMHC(GT), C6H6(GT), NOx(GT), NO2(GT), ..."
3,co_related,32,"CO(GT), PT08.S1(CO), T, RH, AH, Year, Month, D..."
4,nmhc_related,32,"NMHC(GT), PT08.S2(NMHC), T, RH, AH, Year, Mont..."
5,nox_related,32,"NOx(GT), PT08.S3(NOx), T, RH, AH, Year, Month,..."
6,no2_related,32,"NO2(GT), PT08.S4(NO2), T, RH, AH, Year, Month,..."
7,benzene_related,32,"C6H6(GT), PT08.S2(NMHC), T, RH, AH, Year, Mont..."


# 3. 工具函数
## 3.1 评估聚类情况

In [5]:
def evaluate_clustering(X_eval, labels, sample_size=2000):
    labels = np.asarray(labels)
    unique_labels = set(labels)

    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    n_noise = int(np.sum(labels == -1))
    noise_ratio = float(np.mean(labels == -1)) if -1 in unique_labels else 0.0

    result = {
        "n_clusters_found": n_clusters,
        "n_noise": n_noise,
        "noise_ratio": noise_ratio,
        "silhouette": np.nan,
        "calinski_harabasz": np.nan,
        "davies_bouldin": np.nan
    }

    if n_clusters < 2:
        return result

    if -1 in unique_labels:
        mask = labels != -1
        X_eval_valid = X_eval[mask]
        labels_valid = labels[mask]
    else:
        X_eval_valid = X_eval
        labels_valid = labels

    if len(set(labels_valid)) < 2:
        return result

    sample_n = min(sample_size, len(X_eval_valid))

    try:
        result["silhouette"] = silhouette_score(
            X_eval_valid,
            labels_valid,
            sample_size=sample_n,
            random_state=RANDOM_STATE
        )
    except:
        pass

    try:
        result["calinski_harabasz"] = calinski_harabasz_score(X_eval_valid, labels_valid)
    except:
        pass

    try:
        result["davies_bouldin"] = davies_bouldin_score(X_eval_valid, labels_valid)
    except:
        pass

    return result

## 3.2 标准化和pca

In [6]:
def preprocess_feature_set(X_df):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_df)

    pca = PCA(n_components=PCA_VAR_RATIO, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X_scaled)

    return {
        "scaler": scaler,
        "pca_model": pca,
        "X_scaled": X_scaled,
        "X_pca": X_pca,
        "pca_dim": X_pca.shape[1],
        "explained_variance": pca.explained_variance_ratio_.sum()
    }

In [7]:
def build_model_and_labels(algorithm, params_dict, X_input):
    if algorithm == "DBSCAN":
        model = DBSCAN(
            eps=params_dict["eps"],
            min_samples=params_dict["min_samples"],
            n_jobs=-1
        )
        labels = model.fit_predict(X_input)

    elif algorithm == "Hierarchical":
        linkage = params_dict["linkage"]
        if linkage == "ward":
            model = AgglomerativeClustering(
                n_clusters=params_dict["n_clusters"],
                linkage=linkage
            )
        else:
            model = AgglomerativeClustering(
                n_clusters=params_dict["n_clusters"],
                linkage=linkage,
                metric="euclidean"
            )
        labels = model.fit_predict(X_input)

    elif algorithm == "Spectral":
        model = SpectralClustering(
            n_clusters=params_dict["n_clusters"],
            affinity="nearest_neighbors",
            n_neighbors=params_dict["n_neighbors"],
            assign_labels="kmeans",
            random_state=RANDOM_STATE
        )
        labels = model.fit_predict(X_input)

    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")

    return labels

In [8]:
def parse_params_string(algorithm, params_str):
    params = {}
    for item in params_str.split(","):
        key, value = item.strip().split("=")
        key = key.strip()
        value = value.strip()

        if algorithm == "DBSCAN":
            if key == "eps":
                params[key] = float(value)
            elif key == "min_samples":
                params[key] = int(value)

        elif algorithm == "Hierarchical":
            if key == "n_clusters":
                params[key] = int(value)
            elif key == "linkage":
                params[key] = value

        elif algorithm == "Spectral":
            if key == "n_clusters":
                params[key] = int(value)
            elif key == "n_neighbors":
                params[key] = int(value)

    return params

In [9]:
def make_cluster_size_table(labels):
    labels_series = pd.Series(labels, name="cluster_label")
    cluster_size_df = labels_series.value_counts().sort_index().reset_index()
    cluster_size_df.columns = ["cluster_label", "count"]
    cluster_size_df["ratio"] = cluster_size_df["count"] / cluster_size_df["count"].sum()
    return cluster_size_df

In [10]:
def run_clustering_search_for_feature_set(X_df, feature_set_name):
    results = []

    prep = preprocess_feature_set(X_df)
    X_scaled = prep["X_scaled"]
    X_pca = prep["X_pca"]
    pca_dim = prep["pca_dim"]
    explained_var = prep["explained_variance"]

    print(f"Running feature_set = {feature_set_name}")
    print(f"Original dim = {X_df.shape[1]}, PCA dim = {pca_dim}, Explained variance = {explained_var:.4f}")

    # ---------------- DBSCAN ----------------
    min_samples_base = max(5, 2 * pca_dim)

    nbrs = NearestNeighbors(n_neighbors=min_samples_base)
    nbrs.fit(X_pca)
    distances, _ = nbrs.kneighbors(X_pca)
    k_distances = np.sort(distances[:, -1])

    eps_candidates = sorted(set([
        round(np.quantile(k_distances, q), 2)
        for q in [0.90, 0.92, 0.95, 0.97, 0.98]
    ]))

    dbscan_param_grid = {
        "eps": eps_candidates,
        "min_samples": sorted(set([
            max(5, pca_dim),
            min_samples_base,
            min_samples_base + 5
        ]))
    }

    for eps in dbscan_param_grid["eps"]:
        for min_samples in dbscan_param_grid["min_samples"]:
            try:
                params_dict = {"eps": eps, "min_samples": min_samples}
                labels = build_model_and_labels("DBSCAN", params_dict, X_pca)
                metrics = evaluate_clustering(X_pca, labels)

                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "DBSCAN",
                    "params": f"eps={eps}, min_samples={min_samples}",
                    **metrics
                })
            except Exception as e:
                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "DBSCAN",
                    "params": f"eps={eps}, min_samples={min_samples}",
                    "n_clusters_found": np.nan,
                    "n_noise": np.nan,
                    "noise_ratio": np.nan,
                    "silhouette": np.nan,
                    "calinski_harabasz": np.nan,
                    "davies_bouldin": np.nan,
                    "error": str(e)
                })

    # ---------------- Hierarchical ----------------
    hier_param_grid = {
        "n_clusters": [2, 3, 4, 5, 6, 7, 8],
        "linkage": ["ward", "complete", "average"]
    }

    for n_clusters in hier_param_grid["n_clusters"]:
        for linkage in hier_param_grid["linkage"]:
            try:
                params_dict = {"n_clusters": n_clusters, "linkage": linkage}
                labels = build_model_and_labels("Hierarchical", params_dict, X_pca)
                metrics = evaluate_clustering(X_pca, labels)

                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "Hierarchical",
                    "params": f"n_clusters={n_clusters}, linkage={linkage}",
                    **metrics
                })
            except Exception as e:
                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "Hierarchical",
                    "params": f"n_clusters={n_clusters}, linkage={linkage}",
                    "n_clusters_found": np.nan,
                    "n_noise": np.nan,
                    "noise_ratio": np.nan,
                    "silhouette": np.nan,
                    "calinski_harabasz": np.nan,
                    "davies_bouldin": np.nan,
                    "error": str(e)
                })

    # ---------------- Spectral ----------------
    spectral_param_grid = {
        "n_clusters": [2, 3, 4, 5, 6],
        "n_neighbors": [10, 15, 20, 25]
    }

    for n_clusters in spectral_param_grid["n_clusters"]:
        for n_neighbors in spectral_param_grid["n_neighbors"]:
            try:
                params_dict = {"n_clusters": n_clusters, "n_neighbors": n_neighbors}
                labels = build_model_and_labels("Spectral", params_dict, X_pca)
                metrics = evaluate_clustering(X_pca, labels)

                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "Spectral",
                    "params": f"n_clusters={n_clusters}, n_neighbors={n_neighbors}",
                    **metrics
                })
            except Exception as e:
                results.append({
                    "feature_set": feature_set_name,
                    "n_features": X_df.shape[1],
                    "pca_dim": pca_dim,
                    "explained_variance": explained_var,
                    "algorithm": "Spectral",
                    "params": f"n_clusters={n_clusters}, n_neighbors={n_neighbors}",
                    "n_clusters_found": np.nan,
                    "n_noise": np.nan,
                    "noise_ratio": np.nan,
                    "silhouette": np.nan,
                    "calinski_harabasz": np.nan,
                    "davies_bouldin": np.nan,
                    "error": str(e)
                })

    return pd.DataFrame(results)

In [11]:
all_results = []

for feature_set_name, cols in feature_sets.items():
    X_sub = df[cols].copy()
    result_df = run_clustering_search_for_feature_set(X_sub, feature_set_name)
    all_results.append(result_df)

results_df = pd.concat(all_results, axis=0, ignore_index=True)

results_sorted = results_df.sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(Markdown("## 全部搜索结果 Top 30"))
display(results_sorted.head(30))

Running feature_set = all_features
Original dim = 41, PCA dim = 29, Explained variance = 0.9547
Running feature_set = air_quality_core
Original dim = 13, PCA dim = 6, Explained variance = 0.9542
Running feature_set = air_quality_core_temporal
Original dim = 40, PCA dim = 28, Explained variance = 0.9533
Running feature_set = co_related
Original dim = 32, PCA dim = 27, Explained variance = 0.9737
Running feature_set = nmhc_related
Original dim = 32, PCA dim = 27, Explained variance = 0.9602
Running feature_set = nox_related
Original dim = 32, PCA dim = 27, Explained variance = 0.9749
Running feature_set = no2_related
Original dim = 32, PCA dim = 26, Explained variance = 0.9516


KeyboardInterrupt: 

In [ ]:
best_per_feature_algo = (
    results_df.sort_values(
        by=["silhouette", "calinski_harabasz", "davies_bouldin"],
        ascending=[False, False, True]
    )
    .groupby(["feature_set", "algorithm"], as_index=False)
    .first()
)

display(Markdown("## 每个 Feature Set × Algorithm 的最佳结果"))
display(best_per_feature_algo)

In [ ]:
comparison_table = best_per_feature_algo.pivot_table(
    index="feature_set",
    columns="algorithm",
    values="silhouette",
    aggfunc="first"
)

display(Markdown("## 不同 Feature Set × Algorithm 的 Silhouette 对比"))
display(comparison_table)

In [ ]:
comparison_ch = best_per_feature_algo.pivot_table(
    index="feature_set",
    columns="algorithm",
    values="calinski_harabasz",
    aggfunc="first"
)

comparison_dbi = best_per_feature_algo.pivot_table(
    index="feature_set",
    columns="algorithm",
    values="davies_bouldin",
    aggfunc="first"
)

display(Markdown("## CH 对比"))
display(comparison_ch)

display(Markdown("## DBI 对比"))
display(comparison_dbi)

In [ ]:
best_result_objects = []

for _, row in best_per_feature_algo.iterrows():
    feature_set_name = row["feature_set"]
    algorithm = row["algorithm"]
    params_str = row["params"]

    cols = feature_sets[feature_set_name]
    X_sub = df[cols].copy()

    prep = preprocess_feature_set(X_sub)
    X_scaled = prep["X_scaled"]
    X_pca = prep["X_pca"]

    params_dict = parse_params_string(algorithm, params_str)
    labels = build_model_and_labels(algorithm, params_dict, X_pca)

    cluster_size_df = make_cluster_size_table(labels)

    best_result_objects.append({
        "feature_set": feature_set_name,
        "algorithm": algorithm,
        "params": params_str,
        "metrics_row": row,
        "X_scaled": X_scaled,
        "X_pca": X_pca,
        "labels": labels,
        "cluster_size_df": cluster_size_df
    })

display(Markdown("## 最佳结果对象已准备完成"))
print("共", len(best_result_objects), "个最佳结果")

In [ ]:
def plot_cluster_distribution_on_ax(ax, cluster_size_df, title):
    x = cluster_size_df["cluster_label"].astype(str)
    y = cluster_size_df["count"]

    ax.bar(x, y)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=45)

In [ ]:
def plot_pca_2d_on_ax(ax, X_scaled, labels, title):
    pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
    X_2d = pca_2d.fit_transform(X_scaled)

    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, s=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")

In [ ]:
def plot_tsne_2d_on_ax(ax, X_scaled, labels, title, sample_size=TSNE_SAMPLE_SIZE):
    n = len(X_scaled)

    if n > sample_size:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(n, size=sample_size, replace=False)
        X_plot = X_scaled[idx]
        labels_plot = np.asarray(labels)[idx]
    else:
        X_plot = X_scaled
        labels_plot = np.asarray(labels)

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate="auto",
        init="pca",
        random_state=RANDOM_STATE
    )
    X_tsne = tsne.fit_transform(X_plot)

    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_plot, s=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

In [ ]:
def show_grouped_pca_plots(best_result_objects, algorithm_name, ncols=3):
    objs = [obj for obj in best_result_objects if obj["algorithm"] == algorithm_name]
    n = len(objs)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, obj in zip(axes, objs):
        title = f"{obj['feature_set']}\n{obj['algorithm']}"
        plot_pca_2d_on_ax(ax, obj["X_scaled"], obj["labels"], title)

    for ax in axes[len(objs):]:
        ax.axis("off")

    fig.suptitle(f"PCA 2D Comparison - {algorithm_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_grouped_tsne_plots(best_result_objects, algorithm_name, ncols=3):
    objs = [obj for obj in best_result_objects if obj["algorithm"] == algorithm_name]
    n = len(objs)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, obj in zip(axes, objs):
        title = f"{obj['feature_set']}\n{obj['algorithm']}"
        plot_tsne_2d_on_ax(ax, obj["X_scaled"], obj["labels"], title)

    for ax in axes[len(objs):]:
        ax.axis("off")

    fig.suptitle(f"t-SNE 2D Comparison - {algorithm_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def show_grouped_cluster_distribution_plots(best_result_objects, algorithm_name, ncols=3):
    objs = [obj for obj in best_result_objects if obj["algorithm"] == algorithm_name]
    n = len(objs)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = np.array(axes).reshape(-1)

    for ax, obj in zip(axes, objs):
        title = f"{obj['feature_set']}\n{obj['algorithm']}"
        plot_cluster_distribution_on_ax(ax, obj["cluster_size_df"], title)

    for ax in axes[len(objs):]:
        ax.axis("off")

    fig.suptitle(f"Cluster Size Distribution Comparison - {algorithm_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
display(Markdown("## 最佳结果摘要表"))
display(best_per_feature_algo)

In [ ]:
for algo in ["DBSCAN", "Hierarchical", "Spectral"]:
    display(Markdown(f"## {algo} - Cluster Size Distribution Comparison"))
    show_grouped_cluster_distribution_plots(best_result_objects, algo, ncols=3)

In [ ]:
for algo in ["DBSCAN", "Hierarchical", "Spectral"]:
    display(Markdown(f"## {algo} - PCA 2D Comparison"))
    show_grouped_pca_plots(best_result_objects, algo, ncols=3)

In [ ]:
if RUN_TSNE:
    for algo in ["DBSCAN", "Hierarchical", "Spectral"]:
        display(Markdown(f"## {algo} - t-SNE 2D Comparison"))
        show_grouped_tsne_plots(best_result_objects, algo, ncols=3)

In [ ]:
for obj in best_result_objects:
    display(Markdown(f"### {obj['feature_set']} | {obj['algorithm']}"))
    display(obj["metrics_row"].to_frame().T)
    display(obj["cluster_size_df"])